In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import string
import os
import re
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

BASE_URL = "https://www.mayoclinic.org"
SAVE_DIR = "/content/drive/MyDrive/capstone/data"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

TARGET_SECTIONS = ["Symptoms", "Causes", "Risk factors"]
TARGET_SET = frozenset(TARGET_SECTIONS)

SKIP_CLASSES = [
    "eloqua", "subscription", "requestappt", "modal",
    "dialog", "ad-container", "acces-list",
    "contentbox", "rc-list", "related-content",
    "vertical-links", "dynamic-related-content",
]

STOP_H2_CLASSES = [
    "color-primary-inverse",
    "related-content",
    "vertical-links",
    "dynamic-related-content",
]

MAX_WORKERS = 8
REQUEST_DELAY = 0.15

_session = requests.Session()
_session.headers.update(HEADERS)
_adapter = requests.adapters.HTTPAdapter(
    pool_connections=MAX_WORKERS,
    pool_maxsize=MAX_WORKERS,
    max_retries=2,
)
_session.mount("https://", _adapter)
_session.mount("http://", _adapter)

_cache = {}
_cache_lock = threading.Lock()


def get_disease_links(letter):
    url = f"{BASE_URL}/diseases-conditions/index?letter={letter}"
    resp = _session.get(url, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")

    links = []
    seen = set()

    def _add(name, href):
        if not name or not href:
            return
        full = href if href.startswith("http") else BASE_URL + href
        if "/diseases-conditions/" not in full or "/index" in full:
            return
        name = name.strip().rstrip("--- -").strip()
        if not name:
            return
        key = (name.lower(), full)
        if key not in seen:
            seen.add(key)
            links.append({"name": name, "url": full})

    for el in soup.find_all(class_=re.compile(r"primary-name")):
        name_text = None
        for span in el.find_all("span"):
            cls_s = " ".join(span.get("class", []))
            if "separator" in cls_s:
                continue
            if "__name" in cls_s:
                name_text = span.get_text(strip=True)
                break
        see_a = None
        for a in el.find_all("a", href=True):
            a_cls = " ".join(a.get("class", []))
            a_txt = a.get_text(strip=True)
            if "see" in a_cls.lower() or a_txt.startswith("See"):
                see_a = a
                break
        if name_text and see_a:
            _add(name_text, see_a["href"])

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/diseases-conditions/" not in href or "/index" in href:
            continue
        text = a.get_text(strip=True)
        cls_s = " ".join(a.get("class", []))
        if "see-link" in cls_s or "see_link" in cls_s or text.startswith("See "):
            target = text[4:].strip() if text.startswith("See ") else text
            _add(target, href)
            continue
        _add(text, href)

    return links


def _has_skip_ancestor(elem):
    p = elem.parent
    while p and p.name:
        cls_lower = " ".join(p.get("class", [])).lower()
        if cls_lower and any(s in cls_lower for s in SKIP_CLASSES):
            return True
        p = p.parent
    return False


def extract_section_text(h2_tag):
    parts, seen = [], set()
    for elem in h2_tag.find_all_next():
        if elem.name == "h2":
            if elem.get_text(strip=True) in TARGET_SET:
                break
            cls = " ".join(elem.get("class", []))
            if any(f in cls for f in STOP_H2_CLASSES):
                break
            continue
        if elem.name in ("button", "script", "style", "nav", "footer",
                          "header", "input", "label", "form", "select",
                          "svg", "img", "iframe", "noscript"):
            continue
        if elem.name not in ("p", "h3", "h4", "ul", "ol"):
            continue
        if _has_skip_ancestor(elem):
            continue
        txt = elem.get_text(strip=True)
        if txt and txt not in seen:
            seen.add(txt)
            parts.append(txt)
    return " | ".join(parts)


def scrape_disease(url):
    with _cache_lock:
        if url in _cache:
            return _cache[url]
    try:
        resp = _session.get(url, timeout=30, allow_redirects=True)
        time.sleep(REQUEST_DELAY)
        if resp.status_code != 200:
            with _cache_lock:
                _cache[url] = None
            return None
    except requests.RequestException:
        with _cache_lock:
            _cache[url] = None
        return None

    soup = BeautifulSoup(resp.text, "html.parser")
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""
    if not title:
        with _cache_lock:
            _cache[url] = None
        return None

    data = {s: "" for s in TARGET_SECTIONS}
    for h2 in soup.find_all("h2"):
        heading = h2.get_text(strip=True)
        if heading not in TARGET_SET:
            continue
        cls = " ".join(h2.get("class", []))
        if any(f in cls for f in STOP_H2_CLASSES):
            continue
        text = extract_section_text(h2)
        if text and not data[heading]:
            data[heading] = text

    result = {
        "disease_name": title,
        "symptoms":     data["Symptoms"],
        "causes":       data["Causes"],
        "risk_factors": data["Risk factors"],
    }
    with _cache_lock:
        _cache[url] = result
    return result


def _scrape_one(link):
    result = scrape_disease(link["url"])
    return link, result


os.makedirs(SAVE_DIR, exist_ok=True)

print("Collecting disease links...")
all_links = []
for letter in tqdm(string.ascii_uppercase, desc="Indexing"):
    all_links.extend(
        [{"letter": letter, **d} for d in get_disease_links(letter)]
    )
    time.sleep(0.1)
print(f"Total entries found: {len(all_links)}")

unique_urls = set()
unique_links = []
for link in all_links:
    if link["url"] not in unique_urls:
        unique_urls.add(link["url"])
        unique_links.append(link)

print(f"Scraping {len(unique_links)} unique pages...")

results_map = {}
failed_urls = set()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(_scrape_one, lk): lk for lk in unique_links}
    with tqdm(total=len(futures), desc="Scraping") as pbar:
        for fut in as_completed(futures):
            link, result = fut.result()
            if result and result["disease_name"]:
                results_map[link["url"]] = result
            else:
                failed_urls.add(link["url"])
            pbar.update(1)

all_results = []
for link in all_links:
    result = results_map.get(link["url"])
    if result:
        entry = result.copy()
        all_results.append(entry)

df = pd.DataFrame(all_results) if all_results else pd.DataFrame()
df = df.drop_duplicates(subset="disease_name")
output_path = os.path.join(SAVE_DIR, "mayo_clinic_diseases_raw.csv")
df.to_csv(output_path, index=False)

print(f"Done. Saved {len(df)} unique diseases to {output_path}")
print(f"Columns: {df.columns.tolist()}")
print(f"Failed pages: {len(failed_urls)}")
for col in ["symptoms", "causes", "risk_factors"]:
    filled = (df[col] != "").sum()
    print(f"  {col}: {filled}/{len(df)} filled")

Indexing: 100%|██████████| 26/26 [00:11<00:00,  2.23it/s]


Total entries found: 1937
Scraping 1186 unique pages...


Scraping: 100%|██████████| 1186/1186 [03:47<00:00,  5.22it/s]


Done. Saved 1177 unique diseases to /content/drive/MyDrive/capstone/data/mayo_clinic_diseases_raw.csv
Columns: ['disease_name', 'symptoms', 'causes', 'risk_factors']
Failed pages: 0
  symptoms: 1168/1177 filled
  causes: 1164/1177 filled
  risk_factors: 1107/1177 filled


In [5]:
import pandas as pd
import time
import os
from openai import OpenAI
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

SAVE_DIR = "/content/drive/MyDrive/capstone/data"
INPUT_FILE = os.path.join(SAVE_DIR, "mayo_clinic_diseases_raw.csv")
OUTPUT_FILE = os.path.join(SAVE_DIR, "mayo_clinic_diseases_final.csv")
CHECKPOINT_FILE = os.path.join(SAVE_DIR, "cleaning_checkpoint.csv")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)

MODEL = "meta/llama-3.3-70b-instruct"
MAX_WORKERS = 30
CHECKPOINT_EVERY = 50

_done_lock = threading.Lock()

# ================================================================
# DRUG NAMES TO REMOVE
# ================================================================
REMOVE_EXACT = {
    "lithium", "prednisone", "methotrexate", "bleomycin", "gemcitabine",
    "sulfasalazine", "tamoxifen", "ibuprofen", "naproxen sodium",
    "metformin", "warfarin", "acetaminophen", "cyclosporine", "tacrolimus",
    "azathioprine", "rituximab", "adalimumab", "infliximab", "cisplatin",
    "carboplatin", "doxorubicin", "vincristine", "fluorouracil", "capecitabine",
    "paclitaxel", "docetaxel", "etoposide", "cyclophosphamide", "busulfan",
    "melphalan", "chlorambucil", "lomustine", "carmustine", "temozolomide",
    "sirolimus", "everolimus", "hydroxychloroquine", "leflunomide",
    "mycophenolate", "filgrastim", "epoetin", "bortezomib", "carfilzomib",
    "ixazomib", "pomalidomide", "daratumumab", "elotuzumab", "panobinostat",
    "vorinostat", "thalidomide", "lenalidomide", "penicillin", "amoxicillin",
    "erythromycin", "tetracycline", "ciprofloxacin", "doxycycline",
    "interferon", "interleukin", "erythropoietin",
    "androgens", "testosterone", "estrogen", "progesterone", "estrogens",
    "nsaids", "statins", "amphetamines", "barbiturates",
}

REMOVE_FROM_SYMPTOMS_CONTAINING = [
    "insulin", "testosterone", "estrogen", "medication", "medicine",
    "drug", "steroid", "antibiotic", "therapy",
]


# ================================================================
# LLM CALL
# ================================================================
def call_llm(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                top_p=0.7,
                max_tokens=1024,
                stream=False,
            )
            result = resp.choices[0].message.content.strip()

            lines = result.split('\n')
            if len(lines) > 1:
                for line in lines:
                    if '|' in line:
                        result = line.strip()
                        break
                else:
                    result = lines[0].strip()

            if len(result.split()) > 100 and '|' not in result:
                return ""

            keywords = [k.strip() for k in result.split('|') if k.strip()]
            keywords = list(dict.fromkeys(keywords))
            return ' | '.join(keywords)

        except Exception as e:
            err = str(e)
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                print(f"  [ERROR] {err[:80]}")
                return ""
    return ""


# ================================================================
# LLM EXTRACTION
# ================================================================
def extract_symptoms(symptoms_raw, causes_raw, disease_name):
    text = ""
    if symptoms_raw:
        text += f"SYMPTOMS SECTION:\n{symptoms_raw}\n\n"
    if causes_raw:
        text += f"CAUSES SECTION:\n{causes_raw}"
    if not text.strip():
        return ""

    prompt = f"""You are a medical keyword extractor.

From the text below, extract ONLY symptoms of {disease_name} that a patient would use to describe how they feel.

RULES:
- Extract only symptoms that are actually mentioned in the text below
- Do NOT invent or add symptoms that are not in the text
- Do NOT copy examples from these instructions
- Include body location when part of the symptom
- Use simple everyday words a patient would say
- Exclude the disease name: {disease_name}
- Exclude doctor names, medication names, procedure names
- Exclude risk factors like smoking or family history
- Exclude advice or explanatory sentences

FORMAT: Return ONLY keywords from the text separated by | on a single line. Nothing else.

TEXT:
{text}"""

    return call_llm(prompt)


def extract_risk_factors(risk_factors_raw, causes_raw, disease_name):
    text = ""
    if risk_factors_raw:
        text += f"RISK FACTORS SECTION:\n{risk_factors_raw}\n\n"
    if causes_raw:
        text += f"CAUSES SECTION:\n{causes_raw}"
    if not text.strip():
        return ""

    prompt = f"""You are a medical keyword extractor.

From the text below, extract ONLY risk factors and causes of {disease_name}.

RULES:
- Extract only risk factors and causes that are actually mentioned in the text below
- Do NOT invent or add factors that are not in the text
- Do NOT copy examples from these instructions
- Include pre-existing conditions, habits, exposures, family factors, infections, immune factors, demographics like age or gender
- Use simple everyday words a patient would understand
- Exclude the disease name: {disease_name}
- Exclude symptoms like pain, fever, rash
- Exclude specific medication names like lithium, prednisone, methotrexate. General categories like chemotherapy or steroid use are fine.
- Exclude treatment advice, prevention tips, or nutritional advice

FORMAT: Return ONLY keywords from the text separated by | on a single line. Nothing else.

TEXT:
{text}"""

    return call_llm(prompt)


# ================================================================
# POST-PROCESSING
# ================================================================
def clean_symptom_keywords(keywords_str, disease_name):
    if not keywords_str or keywords_str.strip() == "":
        return ""
    disease_lower = disease_name.lower().strip()
    keywords = [k.strip() for k in keywords_str.split('|') if k.strip()]
    cleaned = []
    for kw in keywords:
        kw_lower = kw.lower().strip()
        if kw_lower == disease_lower:
            continue
        if disease_lower in kw_lower and len(kw_lower) < len(disease_lower) + 5:
            continue
        if kw_lower in REMOVE_EXACT:
            continue
        skip = False
        for sub in REMOVE_FROM_SYMPTOMS_CONTAINING:
            if sub in kw_lower:
                skip = True
                break
        if skip:
            continue
        cleaned.append(kw)
    seen = set()
    unique = []
    for kw in cleaned:
        if kw.lower().strip() not in seen:
            seen.add(kw.lower().strip())
            unique.append(kw)
    return ' | '.join(unique)


def clean_rf_keywords(keywords_str, disease_name):
    if not keywords_str or keywords_str.strip() == "":
        return ""
    disease_lower = disease_name.lower().strip()
    keywords = [k.strip() for k in keywords_str.split('|') if k.strip()]
    cleaned = []
    for kw in keywords:
        kw_lower = kw.lower().strip()
        if kw_lower == disease_lower:
            continue
        if disease_lower in kw_lower and len(kw_lower) < len(disease_lower) + 5:
            continue
        if kw_lower in REMOVE_EXACT:
            continue
        cleaned.append(kw)
    seen = set()
    unique = []
    for kw in cleaned:
        if kw.lower().strip() not in seen:
            seen.add(kw.lower().strip())
            unique.append(kw)
    return ' | '.join(unique)


# ================================================================
# PROCESS ONE DISEASE
# ================================================================
def process_disease(row):
    name = str(row["disease_name"]).strip()
    symptoms_raw = str(row.get("symptoms", "")).strip()
    risk_factors_raw = str(row.get("risk_factors", "")).strip()
    causes_raw = str(row.get("causes", "")).strip()

    if symptoms_raw.lower() == "nan":
        symptoms_raw = ""
    if risk_factors_raw.lower() == "nan":
        risk_factors_raw = ""
    if causes_raw.lower() == "nan":
        causes_raw = ""

    symptoms = extract_symptoms(symptoms_raw, causes_raw, name)
    risk_factors = extract_risk_factors(risk_factors_raw, causes_raw, name)

    symptoms = clean_symptom_keywords(symptoms, name)
    risk_factors = clean_rf_keywords(risk_factors, name)

    return {
        "disease_name": name,
        "symptoms": symptoms,
        "risk_factors": risk_factors,
    }


# ================================================================
# RUN
# ================================================================
df = pd.read_csv(INPUT_FILE).fillna("")
unique = df.drop_duplicates(subset="disease_name").copy()

done = {}
if os.path.exists(CHECKPOINT_FILE):
    cp = pd.read_csv(CHECKPOINT_FILE).fillna("")
    for _, row in cp.iterrows():
        if str(row.get("symptoms", "")).strip() != "":
            done[row["disease_name"]] = row.to_dict()
    print(f"Loaded {len(done)} valid entries from checkpoint")

remaining = unique[~unique["disease_name"].isin(done)]
rows_list = [row for _, row in remaining.iterrows()]

print(f"Total diseases: {len(unique)}")
print(f"Already done: {len(done)}")
print(f"Remaining: {len(rows_list)}")

with tqdm(total=len(rows_list), desc="Cleaning") as pbar:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(process_disease, row): row for row in rows_list}
        for i, fut in enumerate(as_completed(futures)):
            result = fut.result()
            with _done_lock:
                done[result["disease_name"]] = result
            pbar.update(1)
            if (i + 1) % CHECKPOINT_EVERY == 0:
                with _done_lock:
                    cp_df = pd.DataFrame(list(done.values()))
                cp_df.to_csv(CHECKPOINT_FILE, index=False)

df_final = pd.DataFrame(list(done.values()))

before = len(df_final)
df_final = df_final[df_final["symptoms"].str.strip() != ""]
after = len(df_final)

df_final.to_csv(OUTPUT_FILE, index=False)

print(f"Done. Total: {before}, With symptoms: {after}, Dropped: {before - after}")
print(f"Saved to: {OUTPUT_FILE}")
for col in ["symptoms", "risk_factors"]:
    filled = (df_final[col].str.strip() != "").sum()
    avg_kw = df_final[col].apply(lambda x: len([k for k in x.split('|') if k.strip()]) if x.strip() else 0).mean()
    print(f"  {col}: {filled}/{len(df_final)} filled, avg {avg_kw:.1f} keywords")

Loaded 0 valid entries from checkpoint
Total diseases: 1177
Already done: 0
Remaining: 1177


Cleaning: 100%|██████████| 1177/1177 [12:22<00:00,  1.59it/s]

Done. Total: 1177, With symptoms: 1171, Dropped: 6
Saved to: /content/drive/MyDrive/capstone/data/mayo_clinic_diseases_final.csv
  symptoms: 1171/1171 filled, avg 14.8 keywords
  risk_factors: 1170/1171 filled, avg 14.6 keywords


In [9]:
import pandas as pd
import time
import os
from openai import OpenAI
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError
import threading

SAVE_DIR = "/content/drive/MyDrive/capstone/data"
INPUT_FILE = os.path.join(SAVE_DIR, "mayo_clinic_diseases_final_v2.csv")
OUTPUT_FILE = os.path.join(SAVE_DIR, "mayo_clinic_diseases_with_specialists.csv")
CHECKPOINT_FILE = os.path.join(SAVE_DIR, "specialist_checkpoint.csv")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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",
    timeout=15
)

MODEL = "meta/llama-3.3-70b-instruct"
MAX_WORKERS = 30
CHECKPOINT_EVERY = 50

_done_lock = threading.Lock()

VALID_SPECIALISTS = [
    "cardiologist", "dermatologist", "endocrinologist", "gastroenterologist",
    "hematologist", "immunologist", "allergist", "infectious disease specialist",
    "nephrologist", "neurologist", "oncologist", "ophthalmologist", "orthopedist",
    "otolaryngologist", "pediatrician", "psychiatrist", "pulmonologist",
    "rheumatologist", "urologist", "gynecologist", "obstetrician",
    "general surgeon", "vascular surgeon", "neurosurgeon",
    "cardiothoracic surgeon", "plastic surgeon", "hepatologist",
    "proctologist", "dentist", "podiatrist", "emergency medicine",
    "general practitioner",
]

VALID_SET = set(VALID_SPECIALISTS)
SPECIALISTS_STR = ", ".join(sorted(VALID_SPECIALISTS))

ALIASES = {
    "ent": "otolaryngologist",
    "ent specialist": "otolaryngologist",
    "ear nose and throat": "otolaryngologist",
    "ear nose and throat specialist": "otolaryngologist",
    "ear, nose and throat": "otolaryngologist",
    "ear, nose, and throat specialist": "otolaryngologist",
    "orthopedic surgeon": "orthopedist",
    "orthopedic specialist": "orthopedist",
    "ob/gyn": "gynecologist",
    "ob-gyn": "gynecologist",
    "obgyn": "gynecologist",
    "gynecologist/obstetrician": "gynecologist",
    "gastroenterologist/hepatologist": "gastroenterologist",
    "allergist/immunologist": "allergist",
    "hematologist/oncologist": "oncologist",
    "hematologist-oncologist": "oncologist",
    "gp": "general practitioner",
    "internist": "general practitioner",
    "primary care": "general practitioner",
    "primary care physician": "general practitioner",
    "family medicine": "general practitioner",
    "oral surgeon": "dentist",
    "periodontist": "dentist",
    "colorectal surgeon": "proctologist",
    "neonatologist": "pediatrician",
    "perinatologist": "obstetrician",
}


def extract_specialist(raw_text):
    text = raw_text.strip().lower()
    text = text.split('\n')[0].strip()
    text = text.rstrip('.').rstrip(',').strip()
    for prefix in ['a ', 'an ', 'the ', 'answer: ', 'answer - ', 'answer is ']:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    if text in VALID_SET:
        return text
    if text in ALIASES:
        return ALIASES[text]

    for spec in sorted(VALID_SPECIALISTS, key=len, reverse=True):
        if spec in text:
            return spec
    for alias, mapped in ALIASES.items():
        if alias in text:
            return mapped

    return None


def get_specialist(disease_name):
    prompt = f"""What single medical specialist primarily treats "{disease_name}"?

Choose exactly one from this list:
{SPECIALISTS_STR}

Reply with ONLY the specialist name. Nothing else."""

    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=30,
            stream=False,
        )
        raw = resp.choices[0].message.content.strip()
        result = extract_specialist(raw)
        if result:
            return result
        return "general practitioner"
    except Exception:
        return "general practitioner"


def process_one(disease_name):
    specialist = get_specialist(disease_name)
    return disease_name, specialist


# ================================================================
# RUN
# ================================================================
df = pd.read_csv(INPUT_FILE).fillna("")

df['disease_name'] = df['disease_name'].str.lower()
df['symptoms'] = df['symptoms'].str.lower()
df['risk_factors'] = df['risk_factors'].str.lower()

# Delete old broken checkpoint
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print("Deleted old checkpoint")

done = {}
remaining = df['disease_name'].tolist()

print(f"Total diseases: {len(remaining)}")

with tqdm(total=len(remaining), desc="Mapping specialists") as pbar:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(process_one, name): name for name in remaining}
        for i, fut in enumerate(as_completed(futures)):
            disease_name, specialist = fut.result()
            with _done_lock:
                done[disease_name] = specialist
            pbar.update(1)
            if (i + 1) % CHECKPOINT_EVERY == 0:
                with _done_lock:
                    cp_df = pd.DataFrame([
                        {"disease_name": d, "specialist": s}
                        for d, s in done.items()
                    ])
                cp_df.to_csv(CHECKPOINT_FILE, index=False)

df['specialist'] = df['disease_name'].map(done).fillna("general practitioner")
df.to_csv(OUTPUT_FILE, index=False)

print(f"\nDone. Saved to {OUTPUT_FILE}")
print(f"\nSpecialist distribution:")
print(df['specialist'].value_counts().to_string())

gp_count = (df['specialist'] == 'general practitioner').sum()
if gp_count > 0:
    print(f"\n{gp_count} diseases defaulted to general practitioner:")
    for d in df[df['specialist'] == 'general practitioner']['disease_name'].tolist()[:20]:
        print(f"  {d}")

Total diseases: 1169


Mapping specialists: 100%|██████████| 1169/1169 [05:16<00:00,  3.70it/s]


Done. Saved to /content/drive/MyDrive/capstone/data/mayo_clinic_diseases_with_specialists.csv

Specialist distribution:
specialist
neurologist                      109
dermatologist                     95
gastroenterologist                84
oncologist                        78
cardiologist                      74
psychiatrist                      62
infectious disease specialist     56
orthopedist                       56
neurosurgeon                      54
endocrinologist                   51
gynecologist                      50
urologist                         42
rheumatologist                    37
otolaryngologist                  34
hematologist                      33
pulmonologist                     32
ophthalmologist                   32
pediatrician                      27
general practitioner              26
dentist                           19
nephrologist                      18
vascular surgeon                  17
allergist                         17
obstetrician     